# 임장 음성메모 PoC — ① 샘플 음성 만들기 + ② 받아쓰기(STT)

**이 노트북이 하는 일**
1. 대본 5개를 AI 목소리(edge-tts)로 읽힌 샘플 녹음(`audio_clean.zip`)을 올린다 → 개인 목소리 없이 누구나 같은 데이터로 재현 가능
2. 현장처럼 **소음을 섞은 버전**도 만든다 (실제 임장은 바람·차 소리가 있다)
3. Whisper 모델 2종 × 깨끗/소음 × 용어 힌트 유무 = 8가지로 받아쓰고 **글자 오류율(CER)·시간**을 비교한다
4. 받아쓴 결과(JSON)를 내려받는다 → 저장소 `data/transcripts/` 폴더에 넣으면 끝

**실행 전**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 선택 후, 위에서부터 ▶ 를 차례로 누르세요. (전체 약 10분)

In [ ]:
!pip -q install openai-whisper jiwer
import torch; print("GPU 사용 가능:", torch.cuda.is_available())

## 1. 대본 (저장소의 `data/scripts.json` 과 같은 내용)

In [ ]:
SCRIPTS = [
 {
  "id": "s1",
  "title": "서울 영등포 — 토허구역(A), 조건 무난",
  "voice": "ko-KR-SunHiNeural",
  "text": "오늘 첫 번째 매물. 서울 영등포구 당산동 아파트, 24평형. 매매는 7억 2천 부르고 전세는 4억 5천 정도 나온대. 2008년식이고 단지는 650세대. 당산역까지 걸어서 8분. 주차는 세대당 1.2대고 엘리베이터 있어. 관리비는 평균 18만원. 베란다 쪽에 누수 흔적이 좀 있어서 확인 필요하고. 남향이라 채광 괜찮고, 초등학교가 단지 바로 옆이라 초품아야. 융자는 없대."
 },
 {
  "id": "s2",
  "title": "인천 부평 — 비대상(B), 갭투자 가능",
  "voice": "ko-KR-InJoonNeural",
  "text": "두 번째는 인천 부평구 부평동 아파트. 32평형이고 매매 5억 3천, 전세는 3억 8천까지 받을 수 있대. 2015년식, 1200세대 대단지. 부평역까지 도보 12분이라 역세권은 좀 애매해. 주차는 세대당 1.4대, 지하주차장 연결돼 있고. 관리비는 25만원 정도. 초등학교는 길 하나 건너야 하고 중학교는 걸어서 10분. 단지 앞에 상가랑 학원가 있어. 관리사무소는 깔끔하게 잘 돌아가는 느낌. 15층 중에 9층, 동향."
 },
 {
  "id": "s3",
  "title": "부천 원미구 빌라 — 비대상(B), 하자 많음",
  "voice": "ko-KR-SunHiNeural",
  "text": "세 번째는 부천 원미구 중동 빌라. 18평. 매매 2억 8천에 전세 2억 3천. 음... 1998년식이라 좀 오래됐어. 엘리베이터 없고 4층인데 계단으로 올라가야 돼. 24세대짜리고 주차는 세대당 0.7대라 빡빡해. 화장실 타일 깨진 데 있고 창틀 쪽에 곰팡이 보였어. 수리비 꽤 들 듯. 대신 역에서 걸어서 5분이라 위치는 좋아. 관리비는 7만원."
 },
 {
  "id": "s4",
  "title": "인천 송도 — 개별 지정(C), 정보 누락 (실패 사례용)",
  "voice": "ko-KR-InJoonNeural",
  "text": "네 번째 본 거는, 그 뭐지, 인천 연수구 송도동 아파트인데 평수는 한 27평쯤 되는 것 같고. 가격은 중개사가 확실히 말 안 해주고 협의 가능하다고만 했어. 전세는 4억 정도라던데. 연식은 잘 모르겠고 한 10년 좀 넘어 보이던데. 일단 다시 물어봐야 될 것 같아."
 },
 {
  "id": "s5",
  "title": "성남 분당 — 토허구역(A), 숫자 정정 (실패 사례용)",
  "voice": "ko-KR-SunHiNeural",
  "text": "마지막 매물, 성남 분당구 정자동. 매매가 9억 9천에 전세 3억 8천. 아, 아니다. 전세는 3억 6천이라고 했던 것 같기도 하고. 평수는 23평. 1995년식 구축이고 세대수는 800세대 정도. 관리비 15만원. 주차는 세대당 0.8대라 좀 빡빡할 듯. 정자역까지 걸어서 6분. 엘리베이터 있고. 융자가 1억 정도 잡혀 있다고 해서 이건 좀 봐야 돼."
 }
]
for s in SCRIPTS: print(s['id'], s['title'])

## 2. AI 목소리 샘플 올리기
음성은 edge-tts(신경망 TTS)로 미리 만들어 두었다 (`data/audio/clean/`, 남·여 목소리 번갈아, 말 속도 +10%).
Colab에서 edge-tts 서버 연결이 막히는 경우가 있어서, 만들어 둔 파일을 올리는 방식으로 바꿨다.

▶ 를 누르면 GitHub 저장소에서 **`audio_clean.zip`** 을 자동으로 내려받는다. (실패하면 파일 선택 버튼이 나온다 → 저장소의 `audio_clean.zip` 선택)

In [ ]:
import os, zipfile, urllib.request
os.makedirs("audio/clean", exist_ok=True)
URL = "https://raw.githubusercontent.com/agogdim99-ux/imjang-voice-memo/main/audio_clean.zip"
try:
    urllib.request.urlretrieve(URL, "audio_clean.zip"); names = ["audio_clean.zip"]
    print("GitHub에서 받음:", URL)
except Exception as e:
    print("자동 다운로드 실패 → 직접 올려 주세요:", e)
    from google.colab import files
    names = list(files.upload())
for name in names:
    with zipfile.ZipFile(name) as z:
        for m in z.namelist():
            if m.endswith(".mp3"):
                open(f"audio/clean/{os.path.basename(m)}", "wb").write(z.read(m))
missing = [s["id"] for s in SCRIPTS if not os.path.exists(f"audio/clean/{s['id']}.mp3")]
print("올라온 파일:", sorted(os.listdir("audio/clean")))
print("빠진 파일:", missing or "없음 ✅")

## 3. 현장 소음 섞기
도로 소음과 비슷한 낮은 주파수 잡음(브라운 노이즈)과 바람 같은 잡음을 섞는다. 신호 대 잡음비(SNR) 10dB = 말소리가 소음보다 약 3배 큰 정도.

In [ ]:
import numpy as np, subprocess, soundfile as sf
os.makedirs("audio/noisy", exist_ok=True)
rng = np.random.default_rng(42)          # 시드 고정 → 누가 돌려도 같은 소음

def load16k(p):
    wav = p.replace(".mp3", ".wav")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", p, "-ac", "1", "-ar", "16000", wav], check=True)
    return sf.read(wav)[0]

def street_noise(n):
    brown = np.cumsum(rng.standard_normal(n)); brown -= np.convolve(brown, np.ones(4000)/4000, "same")
    wind = rng.standard_normal(n) * (0.5 + 0.5*np.sin(np.linspace(0, 20, n)))
    x = brown/np.std(brown) + 0.4*wind/np.std(wind)
    return x/np.std(x)

SNR_DB = 10
for s in SCRIPTS:
    y = load16k(f"audio/clean/{s['id']}.mp3")
    noise = street_noise(len(y)) * np.std(y) / (10 ** (SNR_DB/20))
    out = np.clip(y + noise, -1, 1)
    sf.write(f"audio/noisy/{s['id']}.wav", out, 16000)
    print("저장:", f"audio/noisy/{s['id']}.wav")

In [ ]:
from IPython.display import Audio, display
print("깨끗한 버전"); display(Audio("audio/clean/s1.mp3"))
print("소음 버전");   display(Audio("audio/noisy/s1.wav"))

## 4. Whisper로 받아쓰기 — 8가지 조건 비교
| 비교 축 | 후보 |
|---|---|
| 모델 크기 | `small`(2.4억 파라미터, 빠름) vs `turbo`(large-v3-turbo, 8억, 정확) |
| 음성 | 깨끗 vs 소음 |
| 용어 힌트(initial_prompt) | 없음 vs 부동산 용어 목록 |

**용어 힌트**는 Whisper에 "이런 단어가 나올 수 있다"고 미리 알려주는 설정(하이퍼파라미터)이다. 샘플 지명은 넣지 않고 일반 부동산 용어만 넣어 공정하게 비교한다.

In [ ]:
import whisper, time, json, re, jiwer
PROMPT = "부동산 임장 메모입니다. 매매가, 전세가, 월세, 평형, 연식, 세대수, 세대당 주차, 엘리베이터, 관리비, 융자, 역세권, 초품아, 누수, 곰팡이."

def norm(t):  # 띄어쓰기·문장부호 차이는 오류로 세지 않는다
    return re.sub(r"[\s.,!?…~'\"]", "", t)

os.makedirs("transcripts", exist_ok=True)
rows = []
for model_name in ["small", "turbo"]:
    model = whisper.load_model(model_name)
    for audio in ["clean", "noisy"]:
        for use_prompt in [False, True]:
            tag = f"{model_name}_{audio}_{'prompt' if use_prompt else 'noprompt'}"
            items = []
            for s in SCRIPTS:
                path = f"audio/{audio}/{s['id']}." + ("mp3" if audio == "clean" else "wav")
                t0 = time.time()
                r = model.transcribe(path, language="ko", initial_prompt=PROMPT if use_prompt else None,
                                     temperature=0, fp16=torch.cuda.is_available())
                sec = round(time.time() - t0, 2)
                cer = jiwer.cer(norm(s["text"]), norm(r["text"]))
                items.append({"id": s["id"], "text": r["text"].strip(), "stt_seconds": sec, "cer": round(cer, 4)})
            json.dump({"model": model_name, "audio": audio, "prompt": use_prompt, "items": items},
                      open(f"transcripts/{tag}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
            avg_cer = sum(i["cer"] for i in items)/len(items); avg_sec = sum(i["stt_seconds"] for i in items)/len(items)
            rows.append({"조건": tag, "모델": model_name, "음성": audio, "용어힌트": use_prompt,
                         "평균 CER": round(avg_cer, 4), "1건 평균 시간(초)": round(avg_sec, 2)})
            print(f"{tag:<26} CER {avg_cer:.2%}  {avg_sec:.1f}초/건")
    del model; torch.cuda.empty_cache()

In [ ]:
import pandas as pd
df = pd.DataFrame(rows).sort_values("평균 CER")
df.to_csv("transcripts/stt_results.csv", index=False, encoding="utf-8-sig")
df

## 5. 받아쓴 문장 눈으로 비교 (가장 어려운 조건: 소음 + 힌트 없음)

In [ ]:
for tag in ["small_noisy_noprompt", "turbo_noisy_prompt"]:
    d = json.load(open(f"transcripts/{tag}.json", encoding="utf-8"))
    print(f"\n===== {tag} =====")
    for it in d["items"]:
        print(f"[{it['id']}] CER {it['cer']:.1%}\n  {it['text']}")

## 6. 내려받기
`imjang_stt_results.zip` 이 내려받아진다. 압축을 풀어
- `transcripts/*.json`, `stt_results.csv` → 저장소 `data/transcripts/`
- `audio/` → 저장소 `data/audio/`
에 넣고, Claude Code에 "받아쓰기 결과 넣었어"라고 말하면 된다.

### 6-1. 받아쓰기 결과를 화면에도 출력 (다운로드가 막힐 때 복사해서 쓴다)

In [ ]:
import glob
bundle = {os.path.basename(p): json.load(open(p, encoding="utf-8")) for p in sorted(glob.glob("transcripts/*.json"))}
print("<<<RESULTS_JSON>>>" + json.dumps(bundle, ensure_ascii=False) + "<<<END>>>")
print("<<<CSV>>>" + open("transcripts/stt_results.csv", encoding="utf-8-sig").read() + "<<<END>>>")

### 6-2. zip 내려받기

In [ ]:
!zip -qr imjang_stt_results.zip transcripts audio
from google.colab import files
files.download("imjang_stt_results.zip")